In [1]:
import sys

import numpy as np
from transforms3d.axangles import axangle2mat, mat2axangle
import torch
import plotly.graph_objects as go
import trimesh

from mano_pybullet.hand_model import HandModel20

In [2]:
sys.path.append("..")

from utils.grasp_utils import get_handmodel, rotation_matrix_from_vectors
from model.hand_opt import AdamGraspTransfer

In [3]:
def mat2rvec(mat):
    """Convert rotation matrix to rotation vector."""
    axis, angle = mat2axangle(mat, unit_thresh=1e-05)
    return axis * angle

def rvec2mat(rvec):
    """Convert rotation vector to rotation matrix."""
    angle = np.linalg.norm(rvec)
    axis = rvec if angle != 0.0 else [0.0, 0.0, 1.0]
    mat = axangle2mat(axis, angle)
    return mat

In [4]:
# NOTE: Set the mano hand models dir here. When using with a script, load this directory from a some config file

%env MANO_MODELS_DIR=/home/ninad/Projects/MANO/MANO_Hand_Model/mano_v1_2/models

env: MANO_MODELS_DIR=/home/ninad/Projects/MANO/MANO_Hand_Model/mano_v1_2/models


## Set Sample Data

In [5]:
# Load data for the 00100 frame
# fname = "sample_hamer_output.npz"

# frame_id = "000033"
frame_id = "000172"
# frame_id = "000247"

fname = f"{frame_id}.npz"


data = np.load(f"../data/{fname}", allow_pickle=True)

In [6]:
for k in data.keys():
  print(k)

pred_cam
pred_mano_params
pred_cam_t
focal_length
pred_keypoints_3d
pred_vertices
pred_keypoints_2d
opt_translation
bboxes
right
target_transfer_pose


## Set Left/Right

In [7]:
use_left_hand = True
print("Use left hand? -->", use_left_hand)

rl_index = data['right']
left_idxs = np.arange(rl_index.shape[0])[rl_index==0]
right_idxs = np.arange(rl_index.shape[0])[rl_index==1]

print(left_idxs)
print(right_idxs)

idx_to_use = left_idxs if use_left_hand else right_idxs
print(use_left_hand, idx_to_use)
print(left_idxs.size, right_idxs.size)

Use left hand? --> True
[0]
[]
True [0]
1 0


In [8]:
mano_params = data['pred_mano_params'].item()
print(type(mano_params))
print(mano_params.keys())
print(mano_params['hand_pose'].shape) # for 2 hands

<class 'dict'>
dict_keys(['global_orient', 'hand_pose', 'betas'])
(1, 15, 3, 3)


In [9]:
hand_rotn_mat = mano_params['global_orient'][idx_to_use][0][0].copy()
hand_theta_mat = mano_params['hand_pose'][idx_to_use][0].copy()
mano_trans = data['opt_translation'][idx_to_use][0].copy()
print(hand_rotn_mat.shape)
print(hand_theta_mat.shape)
print(mano_trans.shape)

(3, 3)
(15, 3, 3)
(3,)


In [10]:
# See: https://github.com/geopavlakos/hamer/issues/61#issuecomment-2304863248
# See: https://github.com/geopavlakos/hamer/blob/dc19e5686198a7c3fc3938bff3951f238a85fd11/hamer/datasets/utils.py#L378
if use_left_hand:
  hand_rotn_mat[1::3] *= -1
  hand_rotn_mat[2::3] *= -1
  # hand_theta_mat[1::3] *= -1
  # hand_theta_mat[1::3] *= -1
  pass

In [11]:
hand_theta_full = np.array([mat2rvec(hand_rotn_mat)] + [mat2rvec(hand_theta_mat[i]) for i in range(hand_theta_mat.shape[0])])
print(hand_theta_full.shape, '\n', hand_theta_full)

(16, 3) 
 [[ 1.3892632  -0.9787387  -1.30496187]
 [-0.08111098 -0.16837835  0.43552692]
 [ 0.32834955 -0.00715879  0.63642446]
 [-0.01805465  0.06103654  0.18551041]
 [-0.14338567 -0.13191153  0.67472209]
 [-0.28255698 -0.12133544  0.87322998]
 [ 0.05381811 -0.02634586  0.24424472]
 [ 0.07654206  0.24754745  0.47851342]
 [-0.72277852  0.19620056  0.80778885]
 [-0.1624757   0.07211852  0.45667897]
 [ 0.0718893  -0.07782573  0.59753193]
 [-0.55871122  0.07764017  0.92512846]
 [-0.01783556 -0.02626538  0.4079565 ]
 [ 0.7606787  -0.17281135 -0.53396586]
 [-0.49098646 -0.22610645  0.45486933]
 [ 0.66906766  0.03521193  0.94072805]]


## Init Gripper Models

In [12]:
source_gripper = "mano_left" if use_left_hand else "mano_right"
# source_gripper = "mano_right"

target_gripper = "fetch_gripper"
device = "cpu"
print(device, source_gripper, target_gripper)

cpu mano_left fetch_gripper


In [13]:
source_model = get_handmodel(
  source_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

# sm_left = get_handmodel(
#   "mano_left",
#   1,
#   device,
#   json_path="urdf_assets_meta.json",
#   datadir="../grippers/"
# )


In [14]:
target_model = get_handmodel(
  target_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

## Grasp Pose

In [15]:
# Mano Pybullet Model

hand_model = HandModel20(left_hand=use_left_hand)
# hand_model = HandModel20(left_hand=False)


angles, palm_basis = hand_model.mano_to_angles(hand_theta_full)
# angles, _ = hand_model.mano_to_angles(hand_theta_full)
# angles, palm_basis = hm_left.mano_to_angles(hand_theta_full)

# hm_left = HandModel20(left_hand=True)
# angles_left, palm_basis_left = hm_left.mano_to_angles(hand_theta_full)


# print(len(angles))
# print(palm_basis)

# Reference: https://github.com/kninad/mano_pybullet/blob/960c257cf465f8966e770562b66150beaa359230/mano_pybullet/hand_body.py#L155
origin = hand_model.origins()[0]
# origin = hm_left.origins()[0]

palm_trans = mano_trans + origin - palm_basis @ origin
print(palm_trans)
print(mano_trans)

actual_trans = np.array(palm_trans)
# if use_left_hand:
#   # actual_trans = palm_trans
#   actual_trans -= mano_trans
#   actual_trans[0] *= -1
#   actual_trans += mano_trans

# print(palm_trans)
print(actual_trans)
# palm_trans = np.array([palm_trans[0], mano_trans[1], mano_trans[2]])

[ 0.19976408 -0.16216959  1.9048037 ]
[ 0.27943618 -0.07800826  1.92650055]
[ 0.19976408 -0.16216959  1.9048037 ]


In [16]:
# hand_theta_full[0] = np.zeros(3)

# print(hm_left.mano_to_angles(hand_theta_full)[1])

In [17]:
actual_basis = palm_basis
if use_left_hand:
  R_x = np.array([
      [1, 0, 0],
      [0, -1, 0],
      [0, 0, -1]
  ])
  actual_basis = np.dot(actual_basis, R_x)
# if use_left_hand:
#   print("Updating grasp pose rotn and posn")
#   r_palm_normal = palm_basis @ np.array([0, -1, 0])
#   r_palm_normal_flip = np.array(r_palm_normal)
#   r_palm_normal_flip[0] *= -1
#   rotmat_flip = rotation_matrix_from_vectors(r_palm_normal, r_palm_normal_flip)
#   actual_basis = rotmat_flip @ palm_basis  


In [18]:
# SOURCE GRIPPER (MANO) POSE + DOFS

grasp_pose = torch.zeros(9)
# grasp_pose[0:3] = torch.tensor([0.1, 0.2, 0.3])
# Identity rotation in 6d rot representation is: (1,0,0,0,1,0)
grasp_pose[3:] = torch.tensor(actual_basis.T.reshape(-1)[:6])
grasp_pose[:3] = torch.tensor(actual_trans)
print("Pose:", grasp_pose)

grasp_dofs = -1 * torch.tensor(angles) if use_left_hand else torch.tensor(angles)
# grasp_dofs = torch.tensor(angles)

print("DOFS:", grasp_dofs)

sample_grasp_q = (
  torch.cat(
    [
      grasp_pose,
      grasp_dofs,
    ]
  )
  .unsqueeze(0)
  .to(device)
  .float()
)

Pose: tensor([ 0.1998, -0.1622,  1.9048,  0.1067, -0.9686, -0.2245, -0.0557,  0.2196,
        -0.9740])
DOFS: tensor([-0.1674,  0.4393,  0.6300,  0.1849, -0.0510,  0.6535,  0.8174,  0.2504,
         0.1718,  0.4749,  0.2274,  0.2962, -0.1303,  0.5837,  0.6565,  0.3810,
        -0.7121, -0.4599,  0.5765,  0.2671])


In [19]:
grasp_transfer_opt = AdamGraspTransfer(
  source_gripper,
  target_gripper,
  learning_rate=1e-3,
  device=device
)

q_traj, energy, _ = grasp_transfer_opt.run_adam(
  sample_grasp_q.squeeze(0), running_name="test"
)

min_energy_index = energy.min(dim=0)[1]
print(min_energy_index.item())

print(q_traj.shape)
best_q = q_traj[min_energy_index.item(), -1]
print(best_q.shape)

0
torch.Size([32, 301, 9])
torch.Size([9])


In [20]:
if best_q.shape[0] != 9 + len(target_model.dynamic_joints):
  # We optimized only for pose, so need to provide dummy joints
  best_q = torch.cat((best_q, (target_model.dynamic_joints_q_upper[0] - target_model.dynamic_joints_q_mid[0])), dim=0)

## Viz Src + Target

In [21]:
print("Plotting TARGET and SOURCE together...")

vis_data = source_model.get_plotly_data(q=sample_grasp_q, color='red', opacity=0.2)
target_gripper_mesh_data = target_model.get_plotly_data(q=best_q.unsqueeze(0).float().to(device), color='orange', opacity=0.2)
vis_data += target_gripper_mesh_data
fig = go.Figure(data=vis_data)
fig.show()


Plotting TARGET and SOURCE together...


In [22]:
trimesh_list = []

for mesh in target_gripper_mesh_data:
    vertices = np.array([mesh.x, mesh.y, mesh.z]).T
    faces = np.array([mesh.i, mesh.j, mesh.k]).T
    trimesh_list.append(trimesh.Trimesh(vertices=vertices, faces=faces))

combined_mesh = trimesh.util.concatenate(trimesh_list)
combined_mesh.export(f'../data/target_mesh_{frame_id}_{int(not use_left_hand)}.ply')

b'ply\nformat binary_little_endian 1.0\ncomment https://github.com/mikedh/trimesh\nelement vertex 987\nproperty float x\nproperty float y\nproperty float z\nelement face 1962\nproperty list uchar int vertex_indices\nend_header\n)\x0cn>:\x93\x1f\xbd\xf4o\xec?Y\x85o>\xa0\x8c!\xbd4p\xec?:\xbcj>\xd6+\\\xbd\r&\xeb?\xb1B\x84>\xbcVI\xbd\xbb\x83\xec?\x84A\x84>(\xa6H\xbd\x15\x87\xec?q/\x85>\xfbv\x91\xbd\xd9\xdf\xea?\x86\x88R>\xe1T\x01\xbeY\xab\xe4?\xf0UR>\xe9\xef\x01\xbej\xa4\xe4?S{Q>4\x17\xff\xbd\xcb\xb9\xe4?$\x81e>\xdej\x16\xbe\xb7\xd0\xe5?(\xbc<>\x9a\x1eA\xbe\xec\x87\xef?N\xf2M>\xc0\xfe\x0e\xbe\xfd\xa1\xe5?G\x15U>Q\xc3G\xbek\x89\xef?E\xd6f>\x83\xba\x16\xbe\xe5\xd3\xe5?WT\x81>\x91\xaa\xda\xbdTN\xe8?,\x02|>\x1a\xaf\xf2\xbd\xc5\xdc\xe6?wD\x81>3)\xd8\xbd\x92\x19\xe8?\x16B7>(\x82\xa2\xbd/E\xe8?\xdc\xc88>\x81\xcf\x95\xbdO\x0e\xe9?\x10L9><F\x94\xbd\xe1\xe3\xe8?z\xc1;>\xeeK\x86\xbdR\x82\xe9?9\xbd9>\x06\xd0\xaf\xbdqf\xe7?\x8c\x195>t\x94\xe4\xbd\xaf*\xe5?\xf7\x923>\xfey\xe4\xbd\x1a.\xe5?\xb2\xf4*>\x8f

In [23]:
# q_left = sample_grasp_q.clone()
# q_left[0, 9:] = -1 * torch.tensor(angles_left)


# samp = sample_grasp_q.clone()
# samp[0, :9] = 0
# samp[0, 3] = 1
# samp[0, 7] = 1

# spleft = q_left.clone()
# spleft[0, :9] = 0
# spleft[0, 3] = 1
# # spleft[0, 7] = 1

# vis_data = source_model.get_plotly_data(q=samp, color='red', opacity=0.2)
# vis_data += sm_left.get_plotly_data(q=spleft, color='blue', opacity=0.1)

# fig = go.Figure(data=vis_data)
# fig.show()

In [24]:
# l2r_rotmat = axangle2mat(axis=np.array([0, 1, 0]), angle=np.pi)
# r2l_rotmat = np.linalg.inv(l2r_rotmat)

# l2r_rot6d = l2r_rotmat.T.reshape(-1)[:6]
# print(l2r_rot6d.shape)

# q_left = sample_grasp_q.clone()
# q_left[0, 9:] = -1 * torch.tensor(angles_left)


# samp = sample_grasp_q.clone()
# # samp[0, :3] = 0
# # samp[0, 3] = 1
# # samp[0, 7] = 1


# # left_rot6d = l2r_rot6d
# left_rot6d = (palm_basis @ l2r_rotmat).T.reshape(-1)[:6]
# spleft = q_left.clone()
# # spleft[0, :3] = 0
# spleft[0, 3:9] = torch.tensor(left_rot6d).to(device)



# vis_data = source_model.get_plotly_data(q=samp, color='red', opacity=0.2)
# vis_data += sm_left.get_plotly_data(q=spleft, color='blue', opacity=0.1)

# fig = go.Figure(data=vis_data)
# fig.show()

In [25]:
# rotmat_flip = rotation_matrix_from_vectors(r_palm_normal, r_palm_normal_flip)
# rotmat_flip

In [26]:
hand_ply = f"{frame_id}_{int(not use_left_hand)}.ply"
# hand_ply = f"{frame_id}_{1}.ply"

mano_mesh = trimesh.load_mesh(f"../data/{hand_ply}")
print(mano_mesh)

x, y, z = mano_mesh.vertices.T
# i, j, k = mano_mesh.faces.T

vis_data = []
vis_data += source_model.get_plotly_data(q=sample_grasp_q, color='red', opacity=0.1)


# new_g = sample_grasp_q.clone()
# new_rot6d = (rotmat_flip @ palm_basis).T.reshape(-1)[:6]
# new_g[0, 3:9] = torch.tensor(new_rot6d)
# vis_data += source_model.get_plotly_data(q=new_g, color='blue', opacity=0.1)

vis_data += target_gripper_mesh_data

# left_rot6d = (palm_basis @ l2r_rotmat).T.reshape(-1)[:6]
# q_left = sample_grasp_q.clone()
# q_left[0, 9:] = -1 * torch.tensor(angles_left)
# q_left[0, 3:9] = torch.tensor(left_rot6d).to(device)
# vis_data += sm_left.get_plotly_data(q=q_left, color='blue', opacity=0.1)

vis_data += [
        go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers',
            marker=dict(size=2, color='green')
        )
    ]


fig = go.Figure(data=vis_data)
fig.show()
fig.write_html(f"../data/gtransfer_{frame_id}_{int(not use_left_hand)}.html")


<trimesh.PointCloud(vertices.shape=(778, 3), name=`000172_0.ply`)>


## Viz Mano + URDF

In [27]:
# hand_ply = f"{frame_id}_{int(not use_left_hand)}.ply"
# # hand_ply = f"{frame_id}_{1}.ply"

# mano_mesh = trimesh.load_mesh(f"../data/{hand_ply}")
# print(mano_mesh)

# x, y, z = mano_mesh.vertices.T
# # i, j, k = mano_mesh.faces.T

# vis_data = []
# vis_data += source_model.get_plotly_data(q=sample_grasp_q, color='red', opacity=0.2)
# # vis_data += target_gripper_mesh_data

# # left_rot6d = (palm_basis @ l2r_rotmat).T.reshape(-1)[:6]
# # q_left = sample_grasp_q.clone()
# # q_left[0, 9:] = -1 * torch.tensor(angles_left)
# # q_left[0, 3:9] = torch.tensor(left_rot6d).to(device)
# # vis_data += sm_left.get_plotly_data(q=q_left, color='blue', opacity=0.1)

# vis_data += [
#         go.Scatter3d(
#             x=x, y=y, z=z,
#             mode='markers',
#             marker=dict(size=2, color='green')
#         )
#     ]


# fig = go.Figure(data=vis_data)
# fig.show()


In [28]:
verts = np.array(mano_mesh.vertices)
center = np.mean(verts, axis=0)
print(center.shape)

all_verts = verts - mano_trans
# all_verts[:, 0] *= -1
# all_verts += mano_trans

new_grasp = sample_grasp_q.clone()
new_grasp[0, :3] -= torch.tensor(mano_trans)
vis_data = source_model.get_plotly_data(q=new_grasp, color='red', opacity=0.2)

x,y,z = all_verts.T
vis_data += [
        go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers',
            marker=dict(size=2, color='blue')
        )
    ]

# mano_verts = mano_mesh.vertices
# mano_verts = verts - mano_trans
# x,y,z = mano_verts.T
# vis_data += [
#         go.Scatter3d(
#             x=x, y=y, z=z,
#             mode='markers',
#             marker=dict(size=2, color='green')
#         )
#     ]



fig = go.Figure(data=vis_data)
fig.show()




(3,)


In [29]:
# mano_trans

In [30]:
# print(data['right'])
# print(data['opt_translation'][0])
# print(center)


In [31]:
# verts = np.array(mano_mesh.vertices)
# center = np.mean(verts, axis=0)
# print(center.shape)




# cv = verts - center
# cv[:, 0] *= -1
# # cv[:, 1] *= -1
# nv = cv + center
# x, y, z = nv.T


# all_verts = verts - mano_trans
# all_verts[:, 0] *= -1
# all_verts += mano_trans

# x, y, z = all_verts.T
# # vis_data = []
# vis_data = source_model.get_plotly_data(q=sample_grasp_q, color='red', opacity=0.2)


# flip_grasp = sample_grasp_q.clone()
# flip_grasp[0, :3] -= torch.tensor(mano_trans)
# flip_grasp[0, 0] *= -1
# flip_grasp[0, :3] += torch.tensor(mano_trans)

# vis_data = source_model.get_plotly_data(q=flip_grasp, color='orange', opacity=0.5)

# vis_data += [
#         go.Scatter3d(
#             x=x, y=y, z=z,
#             mode='markers',
#             marker=dict(size=2, color='blue')
#         )
#     ]

# x,y,z = mano_mesh.vertices.T
# vis_data += [
#         go.Scatter3d(
#             x=x, y=y, z=z,
#             mode='markers',
#             marker=dict(size=2, color='green')
#         )
#     ]



# fig = go.Figure(data=vis_data)
# fig.show()
# # fig.write_html("gtransfer_test.html")




In [32]:
# print(mano_trans)

In [33]:
# verts = np.array(mano_mesh.vertices)
# center = np.mean(verts, axis=0)
# print(center.shape)

# all_verts = verts - mano_trans
# all_verts[:, 0] *= -1
# # all_verts += mano_trans



# new_grasp = sample_grasp_q.clone()
# new_grasp[0, :3] -= torch.tensor(mano_trans)

# gg_grasp = new_grasp.clone()
# gg_grasp[0, 0] *= -1
# # vis_data = []
# vis_data = source_model.get_plotly_data(q=new_grasp, color='red', opacity=0.2)


# vis_data += source_model.get_plotly_data(q=gg_grasp, color='orange', opacity=0.2)

# x,y,z = all_verts.T
# vis_data += [
#         go.Scatter3d(
#             x=x, y=y, z=z,
#             mode='markers',
#             marker=dict(size=2, color='blue')
#         )
#     ]

# mano_verts = mano_mesh.vertices
# mano_verts = verts - mano_trans
# x,y,z = mano_verts.T
# vis_data += [
#         go.Scatter3d(
#             x=x, y=y, z=z,
#             mode='markers',
#             marker=dict(size=2, color='green')
#         )
#     ]



# fig = go.Figure(data=vis_data)
# fig.show()
# # fig.write_html("gtransfer_test.html")


